# Raw-Only Stimuli Analysis (Validated Sources)

This notebook recreates the analysis pipeline using only validated files from `stimuli_analysis/raw/stimuli_events`.

- Groups analyzed: `grp-07` to `grp-16`
- Preferred source for grp-08: recovered file if present
- Participant identifiers remain anonymized (`P1`-`P4`)
- Any outlier or combined-master file is explicitly filtered

## 1. Import Required Libraries

Import the necessary libraries, including pandas, numpy, matplotlib, and seaborn.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", 50)
sns.set_theme(style="whitegrid")

## 2. Load Correct Raw Files

Define the correct file paths and load only the validated raw files into DataFrames, filtering out any incorrect or corrupted files.

In [ ]:
RAW_ROOT = Path(
    r"C:\Users\amodica\OneDrive - GN Store Nord\Documents\Codes\affectai-data-processing\stimuli_analysis\raw\stimuli_events"
)
GROUPS = [f"grp-{i:02d}" for i in range(7, 17)]


def _pick_group_file(group_id: str) -> Path:
    """Pick the authoritative stimuli answers file for a group.

    Preference for grp-08: recovered file if available.
    """
    base = RAW_ROOT / group_id / "beh" / "beh"
    recovered = sorted(base.glob("*stimuli_answers_RECOVERED.tsv"))
    if recovered:
        return recovered[0]

    candidates = sorted(base.glob("*task-T0T1T2T3T4_stimuli_answers.tsv"))
    if not candidates:
        raise FileNotFoundError(f"No stimuli file found for {group_id}")

    # Filter likely combined-master outliers by row count.
    valid: list[Path] = []
    for path in candidates:
        with path.open("r", encoding="utf-8", errors="ignore") as handle:
            n_rows = sum(1 for _ in handle) - 1
        if n_rows < 2000:
            valid.append(path)

    if valid:
        return valid[0]
    return candidates[0]


def _events_path(group_id: str) -> Path:
    return RAW_ROOT / group_id / f"events_{group_id}.tsv"


records: list[pd.DataFrame] = []
load_rows: list[dict[str, object]] = []

for grp in GROUPS:
    stim_path = _pick_group_file(grp)
    events_path = _events_path(grp)

    stim_df = pd.read_csv(stim_path, sep="\t")
    stim_df["group_id"] = grp
    stim_df["source_file"] = stim_path.name

    records.append(stim_df)

    events_span_min = np.nan
    if events_path.exists():
        ev = pd.read_csv(events_path, sep="\t")
        events_span_min = (ev["onset"].max() - ev["onset"].min()) / 60

    lsl_span_min = (stim_df["lsl_clock"].max() - stim_df["lsl_clock"].min()) / 60

    load_rows.append(
        {
            "group_id": grp,
            "stimuli_file": stim_path.name,
            "stim_rows": len(stim_df),
            "stim_span_min": lsl_span_min,
            "events_span_min": events_span_min,
            "span_diff_min": lsl_span_min - events_span_min if pd.notna(events_span_min) else np.nan,
        }
    )

raw_df = pd.concat(records, ignore_index=True)
load_summary = pd.DataFrame(load_rows).sort_values("group_id")

print(f"Loaded rows: {len(raw_df):,}")
print(f"Groups loaded: {sorted(raw_df['group_id'].unique())}")
load_summary

## 3. Data Cleaning and Validation

Clean the loaded data by handling missing values, removing duplicates, correcting data types, and validating the integrity of the raw files.

In [ ]:
clean_df = raw_df.copy()

# Normalize dtypes
for col in ["wall_clock", "lsl_clock"]:
    clean_df[col] = pd.to_numeric(clean_df[col], errors="coerce")

for col in ["task", "phase", "response_type", "participant", "device_id", "item_key", "item_value", "group_id"]:
    clean_df[col] = clean_df[col].astype("string")

# Remove exact duplicates
n_before = len(clean_df)
clean_df = clean_df.drop_duplicates()
n_after = len(clean_df)

# Keep only expected tasks and participant labels
expected_tasks = {"T0", "T1", "T2", "T3", "T4"}
expected_participants = {"P1", "P2", "P3", "P4"}

invalid_task_rows = clean_df[~clean_df["task"].isin(expected_tasks)]
invalid_participant_rows = clean_df[
    clean_df["participant"].notna() & (~clean_df["participant"].isin(expected_participants))
]

# Build validation summary
validation_summary = pd.DataFrame(
    {
        "metric": [
            "rows_before_dedup",
            "rows_after_dedup",
            "duplicates_removed",
            "missing_wall_clock",
            "missing_lsl_clock",
            "invalid_task_rows",
            "invalid_participant_rows",
        ],
        "value": [
            n_before,
            n_after,
            n_before - n_after,
            int(clean_df["wall_clock"].isna().sum()),
            int(clean_df["lsl_clock"].isna().sum()),
            int(len(invalid_task_rows)),
            int(len(invalid_participant_rows)),
        ],
    }
)

validation_summary

## 4. Exploratory Data Analysis

Perform exploratory data analysis including summary statistics, data distribution checks, and correlation analysis on the cleaned data.

In [ ]:
# Core EDA tables

rows_by_group_task = (
    clean_df.groupby(["group_id", "task"], dropna=False)
    .size()
    .rename("n_rows")
    .reset_index()
    .sort_values(["group_id", "task"])
)

participant_activity = (
    clean_df.groupby(["group_id", "participant"], dropna=False)
    .size()
    .rename("n_rows")
    .reset_index()
    .sort_values(["group_id", "participant"])
)

# Numeric subset for correlation-style checks
numeric_df = clean_df[["wall_clock", "lsl_clock"]].dropna()
corr_df = numeric_df.corr(numeric_only=True)

print("Rows by group/task:")
rows_by_group_task.head(15)

print("\nParticipant activity:")
participant_activity.head(20)

print("\nNumeric correlations:")
corr_df

## 5. Data Visualization

Create visualizations such as histograms, box plots, and scatter plots to better understand the data distributions and relationships.

In [ ]:
# Histogram: rows per group
rows_per_group = clean_df.groupby("group_id").size().rename("n_rows")

plt.figure(figsize=(8, 4))
sns.histplot(rows_per_group, bins=8, kde=False)
plt.title("Distribution of Rows per Group")
plt.xlabel("Rows")
plt.ylabel("Count of Groups")
plt.tight_layout()
plt.show()

# Box plot: lsl span by task within each group
span_task = (
    clean_df.groupby(["group_id", "task"], as_index=False)
    .agg(lsl_min=("lsl_clock", "min"), lsl_max=("lsl_clock", "max"))
)
span_task["lsl_span_min"] = (span_task["lsl_max"] - span_task["lsl_min"]) / 60

plt.figure(figsize=(10, 4))
sns.boxplot(data=span_task, x="task", y="lsl_span_min")
plt.title("LSL Span (minutes) by Task")
plt.xlabel("Task")
plt.ylabel("Span (minutes)")
plt.tight_layout()
plt.show()

# Scatter: wall_clock vs lsl_clock sample
sample = clean_df[["wall_clock", "lsl_clock"]].dropna().sample(min(3000, len(clean_df)), random_state=42)

plt.figure(figsize=(6, 5))
sns.scatterplot(data=sample, x="wall_clock", y="lsl_clock", s=10, alpha=0.5)
plt.title("Wall Clock vs LSL Clock (sample)")
plt.xlabel("wall_clock")
plt.ylabel("lsl_clock")
plt.tight_layout()
plt.show()

## 6. Statistical Analysis

Apply statistical methods to the cleaned data, including descriptive statistics, hypothesis testing, and trend analysis.

In [ ]:
# Descriptive statistics
desc_numeric = clean_df[["wall_clock", "lsl_clock"]].describe().T

# T1 outcome table
selected = clean_df[
    (clean_df["task"] == "T1")
    & (clean_df["item_key"] == "selected_candidate")
    & clean_df["item_value"].notna()
][["group_id", "participant", "item_value", "wall_clock"]].copy()

selected = selected.drop_duplicates()

outcome_counts = (
    selected.groupby(["group_id", "item_value"]).size().rename("n_submissions").reset_index()
)

# Hypothesis-style check: all groups choose Candidate C?
consensus_by_group = (
    selected.groupby("group_id")["item_value"]
    .agg(lambda x: sorted(set(x.dropna().tolist())))
    .rename("unique_choices")
    .reset_index()
)
consensus_by_group["all_candidate_c"] = consensus_by_group["unique_choices"].apply(
    lambda x: x == ["Candidate C"]
)

# Trend analysis: response density over relative session time (per group)
trend_df = clean_df[["group_id", "lsl_clock"]].dropna().copy()
trend_df["group_lsl_min"] = trend_df.groupby("group_id")["lsl_clock"].transform("min")
trend_df["relative_min"] = (trend_df["lsl_clock"] - trend_df["group_lsl_min"]) / 60
trend_df["time_bin_5min"] = (trend_df["relative_min"] // 5).astype(int)
trend_counts = (
    trend_df.groupby(["group_id", "time_bin_5min"]).size().rename("n_rows").reset_index()
)

# Export summary tables
OUT_DIR = Path(
    r"C:\Users\amodica\OneDrive - GN Store Nord\Documents\Codes\affectai-data-processing\stimuli_analysis\task_outcomes"
)
OUT_DIR.mkdir(parents=True, exist_ok=True)

load_summary.to_csv(OUT_DIR / "raw_only_load_summary.csv", index=False)
validation_summary.to_csv(OUT_DIR / "raw_only_validation_summary.csv", index=False)
outcome_counts.to_csv(OUT_DIR / "raw_only_t1_outcome_counts.csv", index=False)
consensus_by_group.to_csv(OUT_DIR / "raw_only_t1_consensus_by_group.csv", index=False)
trend_counts.to_csv(OUT_DIR / "raw_only_response_trend_5min_bins.csv", index=False)

print("Descriptive stats (numeric):")
desc_numeric

print("\nT1 outcome counts:")
outcome_counts.sort_values(["group_id", "item_value"])

print("\nT1 consensus by group:")
consensus_by_group.sort_values("group_id")

print(f"\nSummary tables exported to: {OUT_DIR}")